# PR_3: Text Preprocessing and TF-IDF

1. Text cleaning
2. Lemmatization
3. Stop word removal
4. Label encoding
5. TF-IDF representation
6. Saving outputs

In [7]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\omcho\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\omcho\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\omcho\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [8]:
base_dir = Path.cwd().resolve().parent
data_path = base_dir / 'Dataset' / 'News_dataset.pickle'
output_dir = base_dir / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_pickle(data_path)
print('Dataset shape:', df.shape)
print('Columns:', list(df.columns))

df['text'] = df['Content'].astype(str)
df['category'] = df['Category'].astype(str)

print(df[['text', 'category']].head())

Dataset shape: (2225, 6)
Columns: ['File_Name', 'Content', 'Category', 'Complete_Filename', 'id', 'News_length']
                                                text  category
0  Ad sales boost Time Warner profit\r\n\r\nQuart...  business
1  Dollar gains on Greenspan speech\r\n\r\nThe do...  business
2  Yukos unit buyer faces loan claim\r\n\r\nThe o...  business
3  High fuel prices hit BA's profits\r\n\r\nBriti...  business
4  Pernod takeover talk lifts Domecq\r\n\r\nShare...  business


In [9]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_tokens(text):
    return [w for w in text.split() if w not in stop_words]

def get_lemmas(words):
    return [lemmatizer.lemmatize(w) for w in words]

df['clean_text'] = df['text'].apply(clean_text)
df['tokens'] = df['clean_text'].apply(get_tokens)
df['lemmatized'] = df['tokens'].apply(get_lemmas)
df['processed_text'] = df['lemmatized'].apply(lambda x: ' '.join(x))

print('clean_text sample:')
print(df[['clean_text']].head())
print('\ntokens sample:')
print(df[['tokens']].head())
print('\nlemmatized sample:')
print(df[['lemmatized']].head())

clean_text sample:
                                          clean_text
0  ad sales boost time warner profit quarterly pr...
1  dollar gains on greenspan speech the dollar ha...
2  yukos unit buyer faces loan claim the owners o...
3  high fuel prices hit ba s profits british airw...
4  pernod takeover talk lifts domecq shares in uk...

tokens sample:
                                              tokens
0  [ad, sales, boost, time, warner, profit, quart...
1  [dollar, gains, greenspan, speech, dollar, hit...
2  [yukos, unit, buyer, faces, loan, claim, owner...
3  [high, fuel, prices, hit, ba, profits, british...
4  [pernod, takeover, talk, lifts, domecq, shares...

lemmatized sample:
                                          lemmatized
0  [ad, sale, boost, time, warner, profit, quarte...
1  [dollar, gain, greenspan, speech, dollar, hit,...
2  [yukos, unit, buyer, face, loan, claim, owner,...
3  [high, fuel, price, hit, ba, profit, british, ...
4  [pernod, takeover, talk, lift, domecq, sh

In [10]:
encoder = LabelEncoder()
df['label_id'] = encoder.fit_transform(df['category'])

print('Encoded label sample:')
print(df[['category', 'label_id']].head())

Encoded label sample:
   category  label_id
0  business         0
1  business         0
2  business         0
3  business         0
4  business         0


In [11]:
vectorizer = TfidfVectorizer(max_features=4000)
tfidf_matrix = vectorizer.fit_transform(df['processed_text'])
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

print('TF-IDF shape:', tfidf_matrix.shape)
print('TF-IDF first 2 rows:')
print(tfidf_matrix.toarray()[:2])

TF-IDF shape: (2225, 4000)
TF-IDF first 2 rows:
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [12]:
out_file = output_dir / 'pr3_all_in_one.csv'

main_cols = ['text', 'category', 'clean_text', 'tokens', 'lemmatized', 'processed_text', 'label_id']
final_df = pd.concat([df[main_cols].copy(), tfidf_df.add_prefix('tfidf_')], axis=1)

final_df.to_csv(out_file, index=False)
print('Saved:', out_file)
print('Final shape:', final_df.shape)

Saved: D:\clg\GS Moze\4th year\8th sem\Practical\BE_NLP_72311560F_OM\PR_3\output\pr3_all_in_one.csv
Final shape: (2225, 4007)
